# NCU Spotlight — PAN1/PAN2 Frame Alignment & PSF Analysis

This notebook processes telescope "Spotlight" TIFF captures into a
stacked, star-sharpness (PSF) analysis pipeline:

1. **Convert** raw `.tif` frames to `.fits` files.
2. **Align & stack** each `PAN1`/`PAN2` frame pair (two experimental
   approaches are kept below for reference — a hybrid catalog+phase
   method, and a simpler correlation-only method).
3. **Measure PSF (FWHM)** of bright stars in each stacked image by
   fitting a 2D Gaussian, and summarize the sharpness distribution
   across all pairs.

Each code cell below has English comments explaining what it does.


## Step 1 — Convert TIFF to FITS

In [1]:
# ============================================================
# Step 1: Batch-convert raw TIFF images into FITS format
# ============================================================
# Astronomy analysis tools (astropy, sep, etc.) expect FITS files,
# so we first walk the raw data folder, find every .tif/.tiff file
# (skipping any already-converted "fits_outputs" folder and any
# hidden files/folders), and convert each one into a matching .fits
# file while preserving the original sub-folder structure.

from pathlib import Path
import tifffile
from astropy.io import fits

# Source folder containing the raw TIFF captures.
src_root = Path('./raw_data')
# Destination folder for the converted FITS files (mirrors src_root's layout).
dst_root = src_root / 'fits_outputs'
dst_root.mkdir(parents=True, exist_ok=True)

# Collect all TIFF files under src_root, excluding:
#   - anything already inside the fits_outputs destination folder
#   - hidden files/folders (names starting with '.')
all_tif_files = [
    p for p in src_root.rglob('*')
    if p.is_file()
    and p.suffix.lower() in ('.tif', '.tiff')
    and 'fits_outputs' not in p.relative_to(src_root).parts
    and not any(part.startswith('.') for part in p.relative_to(src_root).parts)
]

converted = 0
failed = 0

# Convert each TIFF to FITS, keeping the same relative sub-path.
for tif_path in sorted(all_tif_files):
    out_path = dst_root / tif_path.relative_to(src_root).parent / f'{tif_path.stem}.fits'
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        image_data = tifffile.imread(str(tif_path))
        # overwrite=True lets us re-run this cell safely.
        fits.PrimaryHDU(image_data).writeto(str(out_path), overwrite=True)
        print(f'CONVERTED: {tif_path.relative_to(src_root)} -> {out_path.relative_to(src_root)}')
        converted += 1
    except Exception as exc:
        print(f'FAILED: {tif_path.relative_to(src_root)} ({exc})')
        failed += 1

print(f'Summary: {converted} converted, {failed} failed')


CONVERTED: 20260525_PAN1.tif -> fits_outputs/20260525_PAN1.fits
CONVERTED: 20260525_PAN2.tif -> fits_outputs/20260525_PAN2.fits
CONVERTED: 20260602_PAN1.tif -> fits_outputs/20260602_PAN1.fits
CONVERTED: 20260602_PAN2.tif -> fits_outputs/20260602_PAN2.fits
CONVERTED: 20260603_PAN1.tif -> fits_outputs/20260603_PAN1.fits
CONVERTED: 20260603_PAN2.tif -> fits_outputs/20260603_PAN2.fits
CONVERTED: 20260622_PAN1.tif -> fits_outputs/20260622_PAN1.fits
CONVERTED: 20260622_PAN2.tif -> fits_outputs/20260622_PAN2.fits
CONVERTED: 20260623_PAN2.tif -> fits_outputs/20260623_PAN2.fits
CONVERTED: 20260702_PAN1.tif -> fits_outputs/20260702_PAN1.fits
CONVERTED: 20260702_PAN2.tif -> fits_outputs/20260702_PAN2.fits
CONVERTED: 20260707_PAN1.tif -> fits_outputs/20260707_PAN1.fits
CONVERTED: 20260707_PAN2.tif -> fits_outputs/20260707_PAN2.fits
CONVERTED: 20260709_PAN1.tif -> fits_outputs/20260709_PAN1.fits
CONVERTED: 20260709_PAN2.tif -> fits_outputs/20260709_PAN2.fits
CONVERTED: 20260716_PAN1.tif -> fits_out

## Step 2a — Hybrid approach (SEP catalog + phase correlation)

Earlier version of the alignment pipeline: tries three shift-estimation methods (bright-star catalog matching, phase correlation, and a fixed fallback), scores each by stack sharpness, and keeps the best one.

In [2]:
# ============================================================
# Step 2a (early approach): Hybrid shift estimation + stacking
# ============================================================
# This cell now uses the shared helpers from ncu_spots_utils.py instead of
# redefining the alignment functions locally.

from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from matplotlib.backends.backend_pdf import PdfPages

from ncu_spots_utils import (
    center_crop,
    detect_sources,
    estimate_shift_catalog,
    estimate_shift_phase,
    get_roots,
    match_stats,
    refine_shift,
    robust_limits,
)

# Keep the same thresholds as the original notebook workflow.
fallback_shift = (-0.5, -0.5)
match_radius = 1.0
min_matches_ok = 2
max_rms_ok = 1.0

fits_root, stack_root = get_roots()
pdf_path = stack_root / 'pan_pair_triptych.pdf'

# Find every PAN1 frame; its matching PAN2 frame is derived by name substitution.
pan1_files = sorted(fits_root.rglob('*PAN1.fits')) if fits_root.exists() else []
combined = 0
missing = 0
failed = 0
cropped = 0
fallback_used = 0

with PdfPages(pdf_path) as pdf:
    for pan1_path in pan1_files:
        pan2_name = re.sub(r'PAN1\.fits$', 'PAN2.fits', pan1_path.name, flags=re.IGNORECASE)
        pan2_path = pan1_path.with_name(pan2_name)

        if not pan2_path.exists():
            print(f'MISSING PAIR: {pan1_path.relative_to(fits_root)} (no PAN2)')
            missing += 1
            continue

        try:
            data1 = fits.getdata(pan1_path).astype(np.float32)
            data2 = fits.getdata(pan2_path).astype(np.float32)
            if data1.ndim < 2 or data2.ndim < 2:
                print(f'INVALID DIMENSION: {pan1_path.name} or {pan2_path.name}')
                failed += 1
                continue

            # If PAN1/PAN2 have different sizes, crop both to their common
            # center-aligned overlap region before comparing/stacking.
            ny = min(data1.shape[0], data2.shape[0])
            nx = min(data1.shape[1], data2.shape[1])
            if data1.shape[:2] != (ny, nx) or data2.shape[:2] != (ny, nx):
                print(
                    f'CROP TO OVERLAP: {pan1_path.name} {data1.shape[:2]} & {pan2_path.name} {data2.shape[:2]} -> {(ny, nx)}'
                )
                cropped += 1

            data1_crop = center_crop(data1, ny, nx)
            data2_crop = center_crop(data2, ny, nx)

            coords1, flux1 = detect_sources(data1_crop)
            coords2, flux2 = detect_sources(data2_crop)

            # Three candidates: catalog / phase / fallback, each with local refinement.
            cands = {
                'catalog': estimate_shift_catalog(coords1, flux1, coords2, flux2),
                'phase': estimate_shift_phase(data1_crop, data2_crop),
                'fallback': fallback_shift,
            }

            evals = {
                name: refine_shift(data1_crop, data2_crop, coords1, coords2, flux1, s)
                for name, s in cands.items()
            }
            method = max(evals, key=lambda k: evals[k]['score'])
            best = evals[method]

            # Require minimum matching quality unless fallback is best by score.
            if method != 'fallback':
                ok = best['n_match'] >= min_matches_ok and np.isfinite(best['rms']) and best['rms'] <= max_rms_ok
                if not ok:
                    method = 'fallback'
                    best = evals['fallback']
                    fallback_used += 1
            else:
                fallback_used += 1

            shift_yx = best['shift']
            n_match = best['n_match']
            rms = best['rms']
            stacked = best['stacked']
            score_gain = best['score'] - evals['fallback']['score']

            out_name = re.sub(r'PAN1\.fits$', 'PAN_stack.fits', pan1_path.name, flags=re.IGNORECASE)
            out_path = stack_root / pan1_path.relative_to(fits_root).parent / out_name
            out_path.parent.mkdir(parents=True, exist_ok=True)

            # Copy the PAN1 header and record alignment/stacking metadata.
            hdr = fits.getheader(pan1_path)
            hdr['HISTORY'] = 'PAN2 aligned by simplified hybrid SEP+phase estimator'
            hdr['SHIFT_Y'] = (shift_yx[0], 'Shift applied to PAN2 along Y (pixel)')
            hdr['SHIFT_X'] = (shift_yx[1], 'Shift applied to PAN2 along X (pixel)')
            hdr['SHFMETH'] = (method, 'Selected shift method')
            hdr['NMATCH'] = (n_match, 'One-to-one matched stars')
            hdr['RMSSHF'] = (float(rms) if np.isfinite(rms) else -1.0, 'Residual RMS, -1 if invalid')
            hdr['SCGAIN'] = (float(score_gain), 'Score gain over fallback')
            hdr['STACKED'] = ('PAN1+PAN2', 'Two-frame stacked product')

            fits.PrimaryHDU(stacked.astype(np.float32), header=hdr).writeto(out_path, overwrite=True)

            # PDF triptych: PAN1, PAN2, STACK
            p1_vmin, p1_vmax = robust_limits(data1_crop)
            p2_vmin, p2_vmax = robust_limits(data2_crop)
            st_vmin, st_vmax = robust_limits(stacked)

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(data1_crop, origin='lower', cmap='gray', vmin=p1_vmin, vmax=p1_vmax)
            axes[0].set_title('PAN1')
            axes[1].imshow(data2_crop, origin='lower', cmap='gray', vmin=p2_vmin, vmax=p2_vmax)
            axes[1].set_title('PAN2')
            axes[2].imshow(stacked, origin='lower', cmap='gray', vmin=st_vmin, vmax=st_vmax)
            axes[2].set_title('STACK')
            for ax in axes:
                ax.set_xlabel('X')
                ax.set_ylabel('Y')

            fig.suptitle(
                f"{pan1_path.stem} | shift=(dy={shift_yx[0]:.3f}, dx={shift_yx[1]:.3f}), "
                f"method={method}, matched={n_match}, rms={rms if np.isfinite(rms) else float('nan'):.3f}"
            )
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

            print(
                f'STACKED: {pan1_path.relative_to(fits_root)} + {pan2_path.relative_to(fits_root)} '
                f'| shift={shift_yx}, method={method}, matched={n_match}, rms={rms:.3f} -> {out_path.relative_to(stack_root.parent)}'
            )
            combined += 1

        except Exception as exc:
            print(f'FAILED: {pan1_path.name} ({exc})')
            failed += 1

print(f'PDF: {pdf_path}')
print(f'Summary: {combined} stacked, {missing} missing pairs, {failed} failed, {cropped} overlap-cropped, {fallback_used} fallback-used')


CROP TO OVERLAP: 20260525_PAN1.fits (67, 70) & 20260525_PAN2.fits (68, 71) -> (67, 70)
STACKED: 20260525_PAN1.fits + 20260525_PAN2.fits | shift=(-1.0, 0.10000000000000098), method=fallback, matched=0, rms=inf -> fits_stacked/20260525_PAN_stack.fits
CROP TO OVERLAP: 20260602_PAN1.fits (63, 62) & 20260602_PAN2.fits (63, 56) -> (63, 56)
STACKED: 20260602_PAN1.fits + 20260602_PAN2.fits | shift=(5.000000190734861, 1.9500000476837158), method=phase, matched=3, rms=0.514 -> fits_stacked/20260602_PAN_stack.fits
CROP TO OVERLAP: 20260603_PAN1.fits (70, 62) & 20260603_PAN2.fits (69, 58) -> (69, 58)
STACKED: 20260603_PAN1.fits + 20260603_PAN2.fits | shift=(-1.149999976158142, 0.800000023841858), method=phase, matched=3, rms=0.311 -> fits_stacked/20260603_PAN_stack.fits
CROP TO OVERLAP: 20260622_PAN1.fits (59, 52) & 20260622_PAN2.fits (49, 40) -> (49, 40)
STACKED: 20260622_PAN1.fits + 20260622_PAN2.fits | shift=(2.7, -1.9999999999999991), method=catalog, matched=2, rms=0.594 -> fits_stacked/202606

## Step 2b/3 — Using only correlation (refined approach)

Simplified alignment: phase correlation only (no star-catalog matching), followed by bright-star PSF (Gaussian FWHM) fitting and a summary distribution plot. This is the version used for the final PSF analysis.

In [5]:
# ============================================================
# Step 3: Correlation-only alignment + PSF (star sharpness) fitting
# ============================================================
# This cell now uses the shared helpers from ncu_spots_utils.py instead of
# redefining the correlation and PSF functions locally.

from pathlib import Path
import re
import sep
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from matplotlib.backends.backend_pdf import PdfPages

from ncu_spots_utils import (
    crop_pair_to_common_center,
    estimate_shift_correlation,
    filter_outliers_iqr,
    fit_psf_on_patch,
    get_roots,
    image_limits,
    measure_psf_for_bright_stars,
    plot_fit_profiles,
    stack_pair,
)

# PSF measurement/label parameters.
sep_thresh_sigma = 3.5          # SEP detection threshold, in units of background RMS
sep_minarea = 5                 # minimum pixel area for a SEP detection
# The pipeline will measure all detected stars by default; use max_labels
# to put a hard cap if you want to bound runtime/memory (None = no cap).
max_labels = None

# PSF outlier-cleaning parameters for the final distribution plot.
outlier_iqr_k = 1.5             # IQR multiplier used to define outlier bounds
min_points_for_iqr = 8          # need at least this many FWHM values to apply IQR filtering

fits_root, stack_root = get_roots()
pdf_path = stack_root / 'pan_pair_triptych.pdf'

# Find every PAN1 frame; its matching PAN2 frame is derived by name substitution.
pan1_files = sorted(fits_root.rglob('*PAN1.fits')) if fits_root.exists() else []

combined = 0
missing = 0
failed = 0
cropped = 0
fallback_used = 0
all_psf_values = []   # collects every fitted FWHM across all pairs, for the final histogram

with PdfPages(pdf_path) as pdf:
    for pan1_path in pan1_files:
        pan2_name = re.sub(r'PAN1\.fits$', 'PAN2.fits', pan1_path.name, flags=re.IGNORECASE)
        pan2_path = pan1_path.with_name(pan2_name)

        if not pan2_path.exists():
            print(f'MISSING PAIR: {pan1_path.relative_to(fits_root)} (no PAN2)')
            missing += 1
            continue

        try:
            data1 = fits.getdata(pan1_path).astype(np.float32)
            data2 = fits.getdata(pan2_path).astype(np.float32)
            if data1.ndim < 2 or data2.ndim < 2:
                print(f'INVALID DIMENSION: {pan1_path.name} or {pan2_path.name}')
                failed += 1
                continue

            if data1.shape[:2] != data2.shape[:2]:
                cropped += 1
                print(f'CROP TO OVERLAP: {pan1_path.name} {data1.shape[:2]} & {pan2_path.name} {data2.shape[:2]}')

            # Align both frames to a common center-cropped size before correlating.
            d1, d2 = crop_pair_to_common_center(data1, data2)

            shift_yx, corr_value, phase_err = estimate_shift_correlation(d1, d2)
            if not np.isfinite(corr_value):
                shift_yx = (-0.5, -0.5)
                corr_value = -1.0
                fallback_used += 1

            stacked = stack_pair(d1, d2, shift_yx)

            # Measure PSF for all detected stars in the stacked image (default behavior).
            psf_list = measure_psf_for_bright_stars(
                stacked,
                thresh=sep_thresh_sigma,
                minarea=sep_minarea,
                measure_all=True,
                max_stars=max_labels,
                half_size=6,
            )

            # Recompute a high-resolution model only for the brightest star (for plotting),
            # to avoid storing huge arrays for every star.
            brightest_psf = None
            if psf_list:
                brightest = max(psf_list, key=lambda p: p.get('flux', -np.inf))
                # Re-run fit with hr_scale=10 to obtain `model_hr` for plotting.
                # Use the same background-subtracted `sub` image to refit.
                finite = np.isfinite(stacked)
                fill = float(np.nanmedian(stacked[finite]))
                img = np.where(finite, stacked, fill).astype(np.float32)
                bkg = sep.Background(img)
                sub = img - bkg
                brightest_hr = fit_psf_on_patch(sub, brightest['x'], brightest['y'], half_size=6, hr_scale=10)
                if brightest_hr is not None:
                    brightest_hr['flux'] = brightest.get('flux', brightest_hr.get('flux', np.nan))
                    brightest_psf = brightest_hr
                else:
                    brightest_psf = brightest

            # collect every star's FWHM into the global list (flattened across images)
            all_psf_values.extend([p['fwhm'] for p in psf_list if np.isfinite(p['fwhm'])])

            out_name = re.sub(r'PAN1\.fits$', 'PAN_stack.fits', pan1_path.name, flags=re.IGNORECASE)
            out_path = stack_root / pan1_path.relative_to(fits_root).parent / out_name
            out_path.parent.mkdir(parents=True, exist_ok=True)
            print(f'Saving stacked FITS: {out_name}')
            # Copy the PAN1 header and record alignment/PSF metadata.
            hdr = fits.getheader(pan1_path)
            hdr['HISTORY'] = 'PAN2 aligned by correlation-only estimator; PSF measured by Gaussian fit (per-star)'
            hdr['SHIFT_Y'] = (shift_yx[0], 'Shift applied to PAN2 along Y (pixel)')
            hdr['SHIFT_X'] = (shift_yx[1], 'Shift applied to PAN2 along X (pixel)')
            hdr['SHFMETH'] = ('correlation', 'Shift method')
            hdr['CORRVAL'] = (float(corr_value), 'Normalized correlation score')
            hdr['PHERR'] = (float(phase_err) if np.isfinite(phase_err) else -1.0, 'Phase correlation error')
            hdr['NPSF'] = (len(psf_list), 'Number of bright stars with Gaussian PSF fit')
            hdr['STACKED'] = ('PAN1+PAN2', 'Two-frame stacked product')
            fits.PrimaryHDU(stacked.astype(np.float32), header=hdr).writeto(out_path, overwrite=True)

            p1_vmin, p1_vmax = image_limits(d1)
            p2_vmin, p2_vmax = image_limits(d2)
            st_vmin, st_vmax = image_limits(stacked)

            # Build a 2x2 page: PAN1, PAN2, STACK+PSF markers, and fit profiles.
            fig = plt.figure(figsize=(12, 8))
            grid = fig.add_gridspec(2, 2)
            ax1 = fig.add_subplot(grid[0, 0])
            ax2 = fig.add_subplot(grid[0, 1])
            ax3 = fig.add_subplot(grid[1, 0])
            profile_spec = grid[1, 1].subgridspec(2, 1, hspace=0.4)

            ax1.imshow(d1, origin='lower', cmap='gray', vmin=p1_vmin, vmax=p1_vmax)
            ax1.set_title('PAN1')
            ax2.imshow(d2, origin='lower', cmap='gray', vmin=p2_vmin, vmax=p2_vmax)
            ax2.set_title('PAN2')
            ax3.imshow(stacked, origin='lower', cmap='gray', vmin=st_vmin, vmax=st_vmax)
            ax3.set_title('STACK + PSF')

            # Mark each fitted star and label it with its measured FWHM.
            for p in psf_list:
                ax3.plot(p['x'], p['y'], marker='o', markersize=4, markerfacecolor='none', markeredgecolor='cyan', linewidth=0.8)
                ax3.text(p['x'] + 0.8, p['y'] + 0.8, f"PSF={p['fwhm']:.2f}px", color='yellow', fontsize=7)

            for ax in (ax1, ax2, ax3):
                ax.set_xlabel('X')
                ax.set_ylabel('Y')

            plot_fit_profiles(fig, profile_spec, brightest_psf)

            median_psf = float(np.median([p['fwhm'] for p in psf_list])) if psf_list else float('nan')
            fig.suptitle(
                f"{pan1_path.stem} | shift=(dy={shift_yx[0]:.3f}, dx={shift_yx[1]:.3f}), "
                f"corr={corr_value:.4f}, psf_n={len(psf_list)}, psf_med={median_psf:.2f}px"
            )
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

            print(
                f'STACKED: {pan1_path.relative_to(fits_root)} + {pan2_path.relative_to(fits_root)} '
                f'| shift={shift_yx}, corr={corr_value:.4f}, psf_n={len(psf_list)}, psf_med={median_psf:.2f} '
                f'-> {out_path.relative_to(stack_root.parent)}'
            )
            combined += 1

        except Exception as exc:
            print(f'FAILED: {pan1_path.name} ({exc})')
            failed += 1

    # Final chart: PSF distribution across all stacked images (outliers removed by IQR rule).
    if len(all_psf_values) > 0:
        fwhm_raw = np.asarray(all_psf_values, dtype=np.float64)
        fwhm_clean, removed_n, low, high = filter_outliers_iqr(
            fwhm_raw,
            k=outlier_iqr_k,
            min_points=min_points_for_iqr,
        )

        medv = float(np.median(fwhm_clean))
        meanv = float(np.mean(fwhm_clean))

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(fwhm_clean, bins=15, color='#4C78A8', edgecolor='white', alpha=0.9)
        ax.axvline(medv, color='orange', linestyle='--', linewidth=1.8, label=f'Median={medv:.2f}px')
        ax.axvline(meanv, color='crimson', linestyle='-.', linewidth=1.8, label=f'Mean={meanv:.2f}px')
        ax.set_title('PSF FWHM Distribution (Outliers Removed)')
        ax.set_xlabel('FWHM [pixel]')
        ax.set_ylabel('Count')
        ax.legend()
        ax.grid(alpha=0.2)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        print(
            f'PSF Distribution (cleaned): N={len(fwhm_clean)}, removed={removed_n}, '
            f'median={medv:.2f}px, mean={meanv:.2f}px'
        )
        if np.isfinite(low) and np.isfinite(high):
            print(f'IQR bounds: [{low:.2f}, {high:.2f}] px')
    else:
        print('PSF Distribution: no valid PSF measurements to plot')

print(f'PDF: {pdf_path}')
print(f'Summary: {combined} stacked, {missing} missing pairs, {failed} failed, {cropped} overlap-cropped, {fallback_used} fallback-used')


CROP TO OVERLAP: 20260525_PAN1.fits (67, 70) & 20260525_PAN2.fits (68, 71)
Saving stacked FITS: 20260525_PAN_stack.fits
STACKED: 20260525_PAN1.fits + 20260525_PAN2.fits | shift=(-1.7500000238418574, -3.499999999999999), corr=0.7888, psf_n=9, psf_med=2.74 -> fits_stacked/20260525_PAN_stack.fits
CROP TO OVERLAP: 20260602_PAN1.fits (63, 62) & 20260602_PAN2.fits (63, 56)
Saving stacked FITS: 20260602_PAN_stack.fits
STACKED: 20260602_PAN1.fits + 20260602_PAN2.fits | shift=(4.8000001907348615, 2.350000047683716), corr=0.9371, psf_n=3, psf_med=1.58 -> fits_stacked/20260602_PAN_stack.fits
CROP TO OVERLAP: 20260603_PAN1.fits (70, 62) & 20260603_PAN2.fits (69, 58)
Saving stacked FITS: 20260603_PAN_stack.fits
STACKED: 20260603_PAN1.fits + 20260603_PAN2.fits | shift=(-0.7499999761581417, 0.6000000238418579), corr=0.9693, psf_n=3, psf_med=1.70 -> fits_stacked/20260603_PAN_stack.fits
CROP TO OVERLAP: 20260622_PAN1.fits (59, 52) & 20260622_PAN2.fits (49, 40)
Saving stacked FITS: 20260622_PAN_stack.fi